# 02: MASK2FORMER SEGMENTATION (MAPILLARY VISTAS)

High-throughput semantic segmentation for 640x640 GSV images using Mask2Former Swin-Large. Optimized for Colab Pro A100 (40GB VRAM).

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)
print(f"Working directory: {BASE_DIR}")

# Install required packages.
!pip install -q transformers accelerate huggingface_hub

## HUGGING FACE AUTHENTICATION

In [ ]:
# Hugging Face authentication.
from huggingface_hub import login

# Get HF token from Colab secrets.
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token = hf_token)
    print("Logged in to Hugging Face.")
except Exception as e:
    print(f"HF login failed: {e}")
    print("Add HF token to Colab Secrets as 'HF_TOKEN' if needed.")

## GPU VERIFICATION

In [ ]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

print("SYSTEM INFO")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")

    # A100 optimizations.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

    print("A100 optimizations enabled (TF32, cuDNN benchmark).")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU available!")

## IMPORT SETUP

In [ ]:
from pathlib import Path
import json
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

## CONFIGURATION

In [ ]:
# Paths.
IMAGE_DIR = Path("data/processing/images")
META_CSV = Path("data/processing/gsv/metadata.csv")
CHECKPOINT_FILE = Path("data/processing/gsv/segmentation_checkpoint.json")

# Image dimensions.
IMG_SIZE = 640

# Processing config for A100 40GB.
# Mask2Former Swin-Large: ~3GB model + ~50MB per image in batch.
# Safe batch size: (40GB - 3GB) / 0.1GB = 370, but use 64-128 for safety.
BATCH_SIZE = 64
NUM_WORKERS = 8
CHECKPOINT_INTERVAL = 500

print("CONFIGURATION")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"I/O workers: {NUM_WORKERS}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")

## LOAD DATA

In [ ]:
df = pd.read_csv(META_CSV)
print(f"Total images in metadata: {len(df):,}")

if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as f:
        processed_uids = set(json.load(f).get("processed_uids", []))
    print(f"Already processed: {len(processed_uids):,}")
else:
    processed_uids = set()
    print("No checkpoint found. Starting fresh.")

## LOAD MODEL

In [ ]:
torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "facebook/mask2former-swin-large-mapillary-vistas-semantic"
print(f"Loading model: {MODEL_ID}")

# Processor - process at native 640x640.
processor = Mask2FormerImageProcessor.from_pretrained(
    MODEL_ID,
    size = {"height": IMG_SIZE, "width": IMG_SIZE},
    do_resize = True,
    do_rescale = True,
    do_normalize = True
)

# Load model in FP16.
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_ID,
    torch_dtype = torch.float16,
    low_cpu_mem_usage = True
)

model = model.to(device).eval()

# Compile for faster inference (PyTorch 2.0+).
if hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode = "reduce-overhead")
        print("Model compiled with torch.compile()")
    except Exception as e:
        print(f"torch.compile() failed: {e}")

# Memory stats.
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9
    print(f"GPU memory: {allocated:.2f} GB allocated, {free:.1f} GB free")

## SEGMENTATION FUNCTIONS

In [ ]:
def load_image_cv2(path):
    """
    Load image using OpenCV (faster than PIL).
    Returns RGB numpy array or None on failure.
    """
    try:
        img = cv2.imread(str(path))
        if img is None:
            return None
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except (cv2.error, OSError) as e:
        return None


def save_mask_cv2(mask, path):
    """
    Save mask using OpenCV.
    """
    try:
        Path(path).parent.mkdir(parents = True, exist_ok = True)
        cv2.imwrite(str(path), mask)
        return True
    except (cv2.error, OSError) as e:
        return False


@torch.inference_mode()
def segment_batch(images):
    """
    Segment a batch of images.

    Args:
        images: List of RGB numpy arrays.

    Returns:
        List of segmentation masks (numpy uint8 arrays).
    """
    # Preprocess.
    inputs = processor(images = images, return_tensors = "pt")
    pixel_values = inputs["pixel_values"].to(device, dtype = torch.float16)

    # Inference with AMP.
    with torch.cuda.amp.autocast(dtype = torch.float16):
        outputs = model(pixel_values = pixel_values)

    # Post-process to semantic masks.
    masks = processor.post_process_semantic_segmentation(
        outputs,
        target_sizes = [(IMG_SIZE, IMG_SIZE)] * len(images)
    )

    # Convert to numpy.
    results = [m.cpu().numpy().astype(np.uint8) for m in masks]

    # Cleanup.
    del pixel_values, outputs, masks

    return results

## PREPARE PROCESSING LIST

In [ ]:
images_to_process = []

for _, row in df.iterrows():
    uid = row["uid"]

    if uid in processed_uids:
        continue

    original_path = Path(row["image_path"])
    if not original_path.exists():
        continue

    district = str(row["district"]).lower().replace("district ", "")
    ward = str(row["ward"]).lower().replace(" ", "_")

    out_dir = IMAGE_DIR / f"district_{district}" / ward / "segmented"
    mask_path = out_dir / f"class_{uid}.png"

    if mask_path.exists():
        processed_uids.add(uid)
        continue

    images_to_process.append({
        "uid": uid,
        "path": original_path,
        "mask_path": mask_path
    })

print(f"Images to process: {len(images_to_process):,}")
print(f"Already done: {len(processed_uids):,}")

# Time estimate.
# Swin-Large on A100: ~15-30 images/second with batch=64.
if images_to_process:
    est_speed = 20  # Conservative estimate.
    est_minutes = len(images_to_process) / est_speed / 60
    print(f"Estimated time: {est_minutes:.1f} minutes (~{est_speed} img/s)")

## RUN SEGMENTATION

In [ ]:
if not images_to_process:
    print("Nothing to process!")
else:
    start_time = time.time()
    total_processed = 0
    failed_uids = []

    num_batches = (len(images_to_process) + BATCH_SIZE - 1) // BATCH_SIZE

    # Thread pool for parallel I/O.
    executor = ThreadPoolExecutor(max_workers = NUM_WORKERS)

    pbar = tqdm(range(num_batches), desc = "Segmenting", unit = "batch")

    for batch_idx in pbar:
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(images_to_process))
        batch_items = images_to_process[start:end]

        # Parallel image loading.
        paths = [item["path"] for item in batch_items]
        images = list(executor.map(load_image_cv2, paths))

        # Filter failed loads.
        valid_items = []
        valid_images = []
        for item, img in zip(batch_items, images):
            if img is not None:
                valid_items.append(item)
                valid_images.append(img)
            else:
                failed_uids.append(item["uid"])

        if not valid_images:
            continue

        try:
            # Segment batch.
            masks = segment_batch(valid_images)

            # Parallel saving.
            save_args = [(mask, item["mask_path"]) for mask, item in zip(masks, valid_items)]
            save_results = list(executor.map(lambda x: save_mask_cv2(x[0], x[1]), save_args))

            # Update tracking.
            for item, saved in zip(valid_items, save_results):
                if saved:
                    processed_uids.add(item["uid"])
                    total_processed += 1
                else:
                    failed_uids.append(item["uid"])

        except torch.cuda.OutOfMemoryError:
            print(f"\nOOM at batch {batch_idx}! Reducing batch and retrying.")
            torch.cuda.empty_cache()
            gc.collect()

            # Process one at a time.
            for item, img in zip(valid_items, valid_images):
                try:
                    masks = segment_batch([img])
                    if save_mask_cv2(masks[0], item["mask_path"]):
                        processed_uids.add(item["uid"])
                        total_processed += 1
                    else:
                        failed_uids.append(item["uid"])
                except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                    failed_uids.append(item["uid"])
                    torch.cuda.empty_cache()

        except Exception as e:
            print(f"\nBatch {batch_idx} error: {e}")
            for item in valid_items:
                failed_uids.append(item["uid"])

        # Progress stats.
        elapsed = time.time() - start_time
        speed = total_processed / elapsed if elapsed > 0 else 0
        remaining = (len(images_to_process) - total_processed) / speed / 60 if speed > 0 else 0

        pbar.set_postfix({
            "speed": f"{speed:.1f}/s",
            "done": f"{total_processed:,}",
            "ETA": f"{remaining:.1f}m"
        })

        # Checkpoint.
        if total_processed % CHECKPOINT_INTERVAL < BATCH_SIZE:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump({"processed_uids": list(processed_uids)}, f)

        # Memory cleanup every 10 batches.
        if batch_idx % 10 == 0:
            torch.cuda.empty_cache()

    executor.shutdown()

    # Final checkpoint.
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"processed_uids": list(processed_uids)}, f)

    # Summary.
    total_time = time.time() - start_time
    final_speed = total_processed / total_time if total_time > 0 else 0

    print("SEGMENTATION COMPLETE")
    print(f"Processed: {total_processed:,} / {len(images_to_process):,}")
    print(f"Failed: {len(failed_uids):,}")
    print(f"Time: {total_time / 60:.1f} minutes")
    print(f"Speed: {final_speed:.1f} images/second")

## VERIFY AND CLEANUP

In [ ]:
# Cleanup.
del model
torch.cuda.empty_cache()
gc.collect()

print("Model unloaded.")

# Count outputs.
total_masks = 0
print("\nMasks per location:")

for district_dir in sorted(IMAGE_DIR.iterdir()):
    if not district_dir.is_dir():
        continue
    for ward_dir in sorted(district_dir.iterdir()):
        if not ward_dir.is_dir():
            continue
        seg_dir = ward_dir / "segmented"
        if seg_dir.exists():
            count = len(list(seg_dir.glob("class_*.png")))
            total_masks += count
            print(f"  {district_dir.name}/{ward_dir.name}: {count:,}")

print(f"\nTotal masks created: {total_masks:,}")

## VISUALIZE SAMPLE

In [ ]:
import matplotlib.pyplot as plt

sample = df.iloc[0]
uid = sample["uid"]
district = str(sample["district"]).lower().replace("district ", "")
ward = str(sample["ward"]).lower().replace(" ", "_")

orig_path = Path(sample["image_path"])
mask_path = IMAGE_DIR / f"district_{district}" / ward / "segmented" / f"class_{uid}.png"

if orig_path.exists() and mask_path.exists():
    fig, ax = plt.subplots(1, 2, figsize = (12, 6))

    ax[0].imshow(Image.open(orig_path))
    ax[0].set_title(f"Original: {uid}")
    ax[0].axis("off")

    mask = np.array(Image.open(mask_path))
    ax[1].imshow(mask, cmap = "nipy_spectral", vmin = 0, vmax = 64)
    ax[1].set_title("Segmentation (65 Mapillary classes)")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()

    # Class distribution.
    unique, counts = np.unique(mask, return_counts = True)
    print("Top 5 classes in sample:")
    for cls, cnt in sorted(zip(unique, counts), key = lambda x: -x[1])[:5]:
        print(f"  Class {cls}: {cnt / mask.size * 100:.1f}%")
else:
    print("Sample not found.")